# Laboratorio 32 - Drawing on Live Camera

Visión Computacional - CUC

Antes de correr:
- cerrar Zoom / Teams / Meet, si otro programa tiene la cámara no abre
- `pip install opencv-python matplotlib numpy` (la normal, la headless no abre ventanas)
- cuando se abre la ventana hay que darle **clic** para que reciba el teclado. Cada ciclo tiene un límite de tiempo por si acaso

## Configuración

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import json
import platform

os.makedirs('resultados', exist_ok=True)

INDICE_CAMARA = 0
LIMITE_SEG = 60     # corte automático de cada ciclo

# aquí voy guardando lo que mido para el informe
datos = {}
def registrar(clave, valor):
    datos[clave] = valor
    with open('resultados/datos_lab32.json', 'w', encoding='utf-8') as f:
        json.dump(datos, f, indent=2, ensure_ascii=False, default=str)

def cerrar_ventanas():
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)   # a veces las ventanas se quedan pegadas si no se hace esto

def ver(img, titulo='', ancho=8):
    plt.figure(figsize=(ancho, ancho * 0.6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def ver_varias(imgs, titulos, ruta=None, columnas=3):
    filas = (len(imgs) + columnas - 1) // columnas
    fig, ax = plt.subplots(filas, columnas, figsize=(5 * columnas, 3.8 * filas))
    ax = np.atleast_1d(ax).ravel()
    for a in ax:
        a.axis('off')
    for a, im, t in zip(ax, imgs, titulos):
        a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        a.set_title(t, fontsize=10)
    plt.tight_layout()
    if ruta:
        plt.savefig(ruta, bbox_inches='tight')
    plt.show()

registrar('equipo', {'so': f'{platform.system()} {platform.release()}',
                     'python': platform.python_version(), 'opencv': cv2.__version__})
datos['equipo']

## 4.1 Conexión con la cámara

In [ ]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("No fue posible abrir la cámara")
else:
    print('Cámara abierta, backend:', cap.getBackendName())

## 4.2 Dimensiones del video

In [ ]:
print('Lo que devuelve get():', cap.get(cv2.CAP_PROP_FRAME_WIDTH), type(cap.get(cv2.CAP_PROP_FRAME_WIDTH)))

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("Ancho:", width)
print("Alto:", height)
registrar('resolucion', [width, height])

Qué pasa si uso las coordenadas como float (sin el int):

In [ ]:
prueba = np.zeros((100, 100, 3), np.uint8)
w_float = cap.get(cv2.CAP_PROP_FRAME_WIDTH)

try:
    cv2.rectangle(prueba, (w_float / 2, 10.0), (50, 50), (0, 0, 255), 2)
except cv2.error as e:
    print('Error con float:', str(e).strip().splitlines()[-1])

print('width / 2  =', width / 2, '-> float')
print('width // 2 =', width // 2, '-> int, este sí sirve como coordenada')

## 4.3 Rectángulo sobre video en vivo

El de la guía. **q para salir.**

In [ ]:
x = width // 2
y = height // 2
w = width // 4
h = height // 4

t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.rectangle(frame, (x, y), (x + w, y + h),
                  color=(0, 0, 255), thickness=4)

    cv2.imshow('frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q') or time.time() - t0 > LIMITE_SEG:
        break

cap.release()
cerrar_ventanas()

cv2.imwrite('resultados/4_3_rectangulo.jpg', frame)
print('rectángulo de', (x, y), 'a', (x + w, y + h))
ver(frame, 'rectángulo de la guía')

### Actividad: cambiar posición, tamaño, color y grosor

En vez de cambiar el código a mano 6 veces, el ciclo va pasando solo por cada variante (3 s cada una) y guarda un frame de cada una. No hay que hacer nada, solo mirar (q para cortar antes).

In [ ]:
variantes = [
    {'nombre': 'guía (centro, rojo, 4)', 'x': width // 2, 'y': height // 2, 'w': width // 4, 'h': height // 4,
     'color': (0, 0, 255), 'grosor': 4},
    {'nombre': 'esquina sup. izq., verde', 'x': 20, 'y': 20, 'w': width // 4, 'h': height // 4,
     'color': (0, 255, 0), 'grosor': 4},
    {'nombre': 'más grande, azul, 2', 'x': width // 8, 'y': height // 8, 'w': width * 3 // 4, 'h': height * 3 // 4,
     'color': (255, 0, 0), 'grosor': 2},
    {'nombre': 'grosor 15, amarillo', 'x': width // 3, 'y': height // 3, 'w': width // 3, 'h': height // 3,
     'color': (0, 255, 255), 'grosor': 15},
    {'nombre': 'relleno (grosor -1), magenta', 'x': width // 3, 'y': height // 3, 'w': width // 4, 'h': height // 4,
     'color': (255, 0, 255), 'grosor': -1},
    {'nombre': 'se sale de la imagen', 'x': width - 80, 'y': height - 60, 'w': width // 4, 'h': height // 4,
     'color': (0, 128, 255), 'grosor': 4},
]
SEG_VARIANTE = 3

cap = cv2.VideoCapture(INDICE_CAMARA)
muestras = []
t0 = time.time()
i_actual = -1
while True:
    ret, frame = cap.read()
    if not ret:
        break
    i = int((time.time() - t0) // SEG_VARIANTE)
    if i >= len(variantes):
        break
    v = variantes[i]

    cv2.rectangle(frame, (v['x'], v['y']), (v['x'] + v['w'], v['y'] + v['h']), v['color'], v['grosor'])
    cv2.putText(frame, v['nombre'], (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.imshow('variantes', frame)

    # guardo un frame por variante, a la mitad de su tiempo
    if i != i_actual and (time.time() - t0) % SEG_VARIANTE > SEG_VARIANTE / 2:
        muestras.append(frame.copy())
        cv2.imwrite(f'resultados/4_3_variante_{i + 1}.jpg', frame)
        i_actual = i

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cerrar_ventanas()
ver_varias(muestras, [v['nombre'] for v in variantes[:len(muestras)]], 'resultados/4_3_variantes.png')

## 4.4 Parámetros de cv2.rectangle()

In [ ]:
v = variantes[0]
filas = [
    ('frame', 'Imagen sobre la que se dibuja', f'frame de la cámara {width}x{height}x3'),
    ('(x,y)', 'Esquina inicial', str((v['x'], v['y']))),
    ('(x+w,y+h)', 'Esquina opuesta', str((v['x'] + v['w'], v['y'] + v['h']))),
    ('color', 'Color en BGR', f"{v['color']} = rojo; también (0,255,0), (255,0,0), (0,255,255), (255,0,255)"),
    ('thickness', 'Grosor del borde', f"{v['grosor']}; también 2, 15 y -1 (relleno)"),
]
for f in filas:
    print(f'{f[0]:<11} | {f[1]:<30} | {f[2]}')
registrar('4_4_parametros', filas)
registrar('4_3_variantes', variantes)

## 4.5 Rectángulo + círculo + etiqueta al mismo tiempo

**q para salir.**

In [ ]:
cap = cv2.VideoCapture(INDICE_CAMARA)
x, y = width // 2, height // 2
w, h = width // 4, height // 4

t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 4)
    cv2.circle(frame, (x, y), 10, (0, 255, 0), -1)              # -1 = relleno
    cv2.putText(frame, 'Centro', (x + 15, y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                (255, 255, 255), 2)
    cv2.putText(frame, 'Region', (x, y + h + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q') or time.time() - t0 > LIMITE_SEG:
        break

cap.release()
cerrar_ventanas()
cv2.imwrite('resultados/4_5_primitivas.jpg', frame)
ver(frame, 'rectángulo + círculo + texto')

## 4.6 y 4.7 Dibujo interactivo con el mouse

El callback de la guía. Solo le agregué una lista para ir anotando los clics y así poder ver después qué pasó.

Qué hacer: **clic** en un punto (sale el punto rojo), **otro clic** (sale el rectángulo), un **tercer clic** (se borra y empieza otro con ese punto), otro clic para cerrarlo, y **q**.

In [ ]:
pt1 = (0, 0)
pt2 = (0, 0)
topLeft_clicked = False
botRight_clicked = False
clics = []   # para el informe

def draw_rectangle(event, x, y, flags, param):
    global pt1, pt2, topLeft_clicked, botRight_clicked

    if event == cv2.EVENT_LBUTTONDOWN:
        clics.append((x, y))

        # si ya había rectángulo, se reinicia
        if topLeft_clicked and botRight_clicked:
            topLeft_clicked = False
            botRight_clicked = False
            pt1 = (0, 0)
            pt2 = (0, 0)

        if not topLeft_clicked:
            pt1 = (x, y)
            topLeft_clicked = True
        elif not botRight_clicked:
            pt2 = (x, y)
            botRight_clicked = True

In [ ]:
cap = cv2.VideoCapture(INDICE_CAMARA)

cv2.namedWindow('Test')
cv2.setMouseCallback('Test', draw_rectangle)

evidencias = {}
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if topLeft_clicked:
        cv2.circle(frame, pt1, 5, (0, 0, 255), -1)

    if topLeft_clicked and botRight_clicked:
        cv2.rectangle(frame, pt1, pt2, (0, 0, 255), 2)

    cv2.imshow('Test', frame)

    # guardo la primera vez que aparece solo el punto y la primera vez que aparece el rectángulo
    if topLeft_clicked and not botRight_clicked and 'punto' not in evidencias:
        evidencias['punto'] = frame.copy()
    if topLeft_clicked and botRight_clicked and 'rectangulo' not in evidencias:
        evidencias['rectangulo'] = frame.copy()

    if cv2.waitKey(1) & 0xFF == ord('q') or time.time() - t0 > LIMITE_SEG:
        break

cap.release()
cerrar_ventanas()

evidencias['final'] = frame.copy()
for k, im in evidencias.items():
    cv2.imwrite(f'resultados/4_7_{k}.jpg', im)
print('clics:', clics)
print('último pt1, pt2:', pt1, pt2)
registrar('4_7_interactivo', {'clics': clics, 'pt1': pt1, 'pt2': pt2})
ver_varias(list(evidencias.values()), list(evidencias.keys()), 'resultados/4_7_interactivo.png')

## 4.9 Experimentación

Todo en un solo callback:
1. el punto del primer clic ahora es más grande (8) y azul
2. texto con las coordenadas de cada clic
3. otros eventos: `EVENT_MOUSEMOVE` para ver una vista previa del rectángulo mientras muevo el mouse, y `EVENT_RBUTTONDOWN` (clic derecho) para borrar todo
4. varios rectángulos (se guardan en una lista)
5. ancho y alto de cada rectángulo

Qué hacer: dibujar **2 o 3 rectángulos** (dos clics cada uno), probar el **clic derecho** para borrar, dibujar uno más, **s** para guardar captura y **q** para salir.

In [ ]:
rects = []            # rectángulos terminados: (p1, p2)
inicio = None         # primer clic del rectángulo que se está haciendo
mouse = None          # posición actual del mouse
eventos = []          # para el informe

def callback_exp(event, x, y, flags, param):
    global inicio, mouse, rects
    if event == cv2.EVENT_LBUTTONDOWN:
        if inicio is None:
            inicio = (x, y)
        else:
            rects.append((inicio, (x, y)))
            inicio = None
        eventos.append(('izq', x, y))
    elif event == cv2.EVENT_RBUTTONDOWN:
        rects, inicio = [], None
        eventos.append(('der', x, y))
    elif event == cv2.EVENT_MOUSEMOVE:
        mouse = (x, y)

cap = cv2.VideoCapture(INDICE_CAMARA)
cv2.namedWindow('Experimentos')
cv2.setMouseCallback('Experimentos', callback_exp)

colores = [(0, 0, 255), (0, 255, 0), (255, 0, 0), (0, 255, 255), (255, 0, 255)]
capturas_exp = []
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break

    for i, (p1, p2) in enumerate(rects):
        c = colores[i % len(colores)]
        cv2.rectangle(frame, p1, p2, c, 2)
        ancho, alto = abs(p2[0] - p1[0]), abs(p2[1] - p1[1])
        esq = (min(p1[0], p2[0]), min(p1[1], p2[1]))
        cv2.putText(frame, f'#{i + 1} {ancho}x{alto}', (esq[0], max(esq[1] - 8, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, c, 2)
        for p in (p1, p2):
            cv2.putText(frame, f'{p}', (p[0] + 6, p[1] + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    if inicio is not None:
        cv2.circle(frame, inicio, 8, (255, 0, 0), -1)       # punto más grande y azul
        cv2.putText(frame, f'{inicio}', (inicio[0] + 10, inicio[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
        if mouse is not None:
            cv2.rectangle(frame, inicio, mouse, (200, 200, 200), 1)   # vista previa

    cv2.putText(frame, f'rectangulos: {len(rects)} | clic der: borrar | s: guardar | q: salir',
                (10, height - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.imshow('Experimentos', frame)

    k = cv2.waitKey(1) & 0xFF
    if k == ord('s'):
        ruta = f'resultados/4_9_captura_{len(capturas_exp) + 1}.jpg'
        cv2.imwrite(ruta, frame)
        capturas_exp.append(ruta)
        print('guardada', ruta)
    if k == ord('q') or time.time() - t0 > LIMITE_SEG:
        break

cap.release()
cerrar_ventanas()
cv2.imwrite('resultados/4_9_final.jpg', frame)

medidas = [{'p1': p1, 'p2': p2, 'ancho': abs(p2[0] - p1[0]), 'alto': abs(p2[1] - p1[1])} for p1, p2 in rects]
for m in medidas:
    print(m)
print('eventos:', eventos)
registrar('4_9_experimentos', {'rectangulos_finales': medidas, 'eventos': eventos, 'capturas': capturas_exp})
ver(frame, 'último frame de los experimentos')

## 5. Actividad integradora - anotador

El mismo código está en `anotador_camara.py` para correrlo desde la terminal.

Qué hacer: seleccionar un objeto con **dos clics**, **s** para guardar, **r** para reiniciar, seleccionar otro, **s** otra vez y **q**.

In [ ]:
# Sistema sencillo de anotación sobre cámara en vivo (actividad integradora Lab 32)
#
# Mouse:  clic izquierdo = primera esquina, segundo clic = esquina opuesta
#         (mientras se mueve el mouse se ve una vista previa de la caja)
# Teclas: r reiniciar selección | s guardar captura | q salir
#
# Uso desde terminal:  python anotador_camara.py        (cámara 0)
#                      python anotador_camara.py 1      (otra cámara)

import os
import sys
import time

import cv2

VENTANA = 'Anotador'
ROJO, VERDE, AMARILLO, BLANCO, NEGRO = (0, 0, 255), (0, 255, 0), (0, 255, 255), (255, 255, 255), (0, 0, 0)

estado = {'p1': None, 'p2': None, 'mouse': None}


def al_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if estado['p1'] is None or estado['p2'] is not None:
            # primer clic (o clic después de tener una caja completa): empieza una nueva
            estado['p1'], estado['p2'] = (x, y), None
        else:
            estado['p2'] = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE:
        estado['mouse'] = (x, y)


def normalizar(p1, p2, w, h):
    # que funcione sin importar en qué orden se hagan los clics, y sin salirse de la imagen
    x1, x2 = sorted((min(max(p1[0], 0), w - 1), min(max(p2[0], 0), w - 1)))
    y1, y2 = sorted((min(max(p1[1], 0), h - 1), min(max(p2[1], 0), h - 1)))
    return x1, y1, x2, y2


def texto_con_fondo(img, texto, org, color_fondo, escala=0.55, grosor=1):
    (tw, th), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, escala, grosor)
    x, y = org
    y = max(y, th + 4)          # que no se salga por arriba
    cv2.rectangle(img, (x, y - th - 4), (x + tw + 4, y + base), color_fondo, -1)
    cv2.putText(img, texto, (x + 2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, escala, NEGRO if color_fondo != NEGRO else BLANCO, grosor)


def anotador(fuente=0, etiqueta='ROI', limite_seg=None, carpeta='resultados/anotador'):
    os.makedirs(carpeta, exist_ok=True)
    estado.update(p1=None, p2=None, mouse=None)
    rois, capturas = [], []

    # 1. validar la cámara
    cap = cv2.VideoCapture(fuente)
    if not cap.isOpened():
        print(f'No fue posible abrir la cámara {fuente}')
        return {'error': 'sin cámara'}
    ok, frame = cap.read()
    if not ok:
        print('La cámara abrió pero no entrega frames')
        cap.release()
        return {'error': 'sin frames'}

    # 2. resolución (del frame real)
    h, w = frame.shape[:2]
    print(f'Cámara {fuente} conectada: {w}x{h}')

    cv2.namedWindow(VENTANA)
    cv2.setMouseCallback(VENTANA, al_mouse)

    ultima_guardada = None
    t0 = time.time()
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print('read() devolvió False')
                break
            vista = frame.copy()

            texto_con_fondo(vista, f'Resolucion: {w}x{h}', (8, 22), BLANCO)
            cv2.putText(vista, 'clic: esquinas | r: reiniciar | s: guardar | q: salir', (8, h - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, BLANCO, 1)

            p1, p2 = estado['p1'], estado['p2']
            if p1 is not None:
                cv2.circle(vista, p1, 5, ROJO, -1)
                if p2 is None and estado['mouse'] is not None:
                    # vista previa mientras se busca la segunda esquina
                    cv2.rectangle(vista, p1, estado['mouse'], AMARILLO, 1)

            if p1 is not None and p2 is not None:
                x1, y1, x2, y2 = normalizar(p1, p2, w, h)
                bw, bh = x2 - x1, y2 - y1

                # 4. bounding box
                cv2.rectangle(vista, (x1, y1), (x2, y2), VERDE, 2)
                # 5. coordenadas de las dos esquinas
                cv2.circle(vista, (x1, y1), 4, ROJO, -1)
                cv2.circle(vista, (x2, y2), 4, ROJO, -1)
                cv2.putText(vista, f'({x1},{y1})', (x1 + 4, y1 + 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, AMARILLO, 1)
                cv2.putText(vista, f'({x2},{y2})', (max(x2 - 85, 0), min(y2 + 16, h - 25)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, AMARILLO, 1)
                # 6 y 7. ancho, alto y etiqueta encima de la región
                texto_con_fondo(vista, f'{etiqueta}  {bw}x{bh} px', (x1, y1 - 4), VERDE)

                caja = (x1, y1, bw, bh)
                if not rois or rois[-1]['caja'] != caja:
                    rois.append({'esquina1': (x1, y1), 'esquina2': (x2, y2), 'caja': caja,
                                 'ancho': bw, 'alto': bh, 'area': bw * bh})
                    print(f'ROI: esquinas ({x1},{y1}) y ({x2},{y2}), ancho={bw}, alto={bh}')

            cv2.imshow(VENTANA, vista)
            k = cv2.waitKey(1) & 0xFF

            if k == ord('q'):
                break
            elif k == ord('r'):
                # 8. reiniciar
                estado.update(p1=None, p2=None)
                print('selección reiniciada')
            elif k == ord('s'):
                ruta = f'{carpeta}/anotacion_{len(capturas) + 1}.jpg'
                cv2.imwrite(ruta, vista)
                capturas.append(ruta)
                print('guardada', ruta)

            ultima_guardada = vista
            if limite_seg and time.time() - t0 > limite_seg:
                print(f'límite de {limite_seg} s')
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        for _ in range(5):
            cv2.waitKey(1)

    if ultima_guardada is not None:
        cv2.imwrite(f'{carpeta}/ultimo_frame.jpg', ultima_guardada)
    return {'resolucion': [w, h], 'rois': rois, 'capturas': capturas}

In [ ]:
resumen_anot = anotador(fuente=INDICE_CAMARA, etiqueta='ROI', limite_seg=90)
registrar('5_integradora', resumen_anot)
print('ROIs seleccionadas:', len(resumen_anot.get('rois', [])))
imgs = [cv2.imread(r) for r in resumen_anot.get('capturas', [])]
if imgs:
    ver_varias(imgs, [os.path.basename(r) for r in resumen_anot['capturas']], 'resultados/5_anotaciones.png', columnas=2)

## 6. Reto - simular la salida de un detector

Hago una función que dibuja una "detección" como lo hacen los detectores (caja + etiqueta con fondo + confianza opcional). Primero la uso con una caja que selecciono yo a mano (dos clics) con la etiqueta 'Objeto seleccionado'.

Qué hacer: seleccionar un objeto con **dos clics**, **s** para guardar, **q** para salir.

In [ ]:
def dibujar_deteccion(img, caja, etiqueta, confianza=None, color=(0, 200, 0)):
    x, y, w, h = [int(v) for v in caja]
    cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)
    texto = etiqueta if confianza is None else f'{etiqueta} {confianza:.2f}'
    (tw, th), base = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_txt = max(y, th + 6)
    cv2.rectangle(img, (x, y_txt - th - 6), (x + tw + 6, y_txt), color, -1)
    cv2.putText(img, texto, (x + 3, y_txt - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

sel = {'p1': None, 'p2': None}
def seleccionar(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if sel['p1'] is None or sel['p2'] is not None:
            sel['p1'], sel['p2'] = (x, y), None
        else:
            sel['p2'] = (x, y)

cap = cv2.VideoCapture(INDICE_CAMARA)
cv2.namedWindow('Reto')
cv2.setMouseCallback('Reto', seleccionar)
caja_manual = None
guardadas_reto = []
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if sel['p1'] is not None and sel['p2'] is None:
        cv2.circle(frame, sel['p1'], 5, (0, 0, 255), -1)
    if sel['p2'] is not None:
        (x1, y1), (x2, y2) = sel['p1'], sel['p2']
        caja_manual = (min(x1, x2), min(y1, y2), abs(x2 - x1), abs(y2 - y1))   # formato (x, y, w, h)
        dibujar_deteccion(frame, caja_manual, 'Objeto seleccionado')
    cv2.imshow('Reto', frame)
    k = cv2.waitKey(1) & 0xFF
    if k == ord('s'):
        ruta = f'resultados/6_objeto_seleccionado_{len(guardadas_reto) + 1}.jpg'
        cv2.imwrite(ruta, frame)
        guardadas_reto.append(ruta)
    if k == ord('q') or time.time() - t0 > LIMITE_SEG:
        break
cap.release()
cerrar_ventanas()
cv2.imwrite('resultados/6_objeto_seleccionado.jpg', frame)
print('caja (x, y, w, h):', caja_manual)
registrar('6_reto_manual', {'caja': caja_manual, 'capturas': guardadas_reto})
ver(frame, 'selección manual con estilo de detector')

### Ahora con un detector de verdad

La misma función `dibujar_deteccion` pero con las cajas que saca un detector de rostros (YuNet, el que trae OpenCV en `cv2.FaceDetectorYN`; el modelo está en `modelos/`). El detector devuelve para cada rostro `x, y, w, h`, 5 puntos de la cara y la confianza, o sea el mismo formato de caja que la selección manual.

Solo hay que mirar a la cámara unos segundos (**q** para salir).

In [ ]:
modelo = 'modelos/face_detection_yunet_2023mar.onnx'
detector = cv2.FaceDetectorYN.create(modelo, '', (width, height), 0.8)

cap = cv2.VideoCapture(INDICE_CAMARA)
n, con_rostro, t_det = 0, 0, 0.0
mejor = None
t0 = time.time()
while True:
    ret, frame = cap.read()
    if not ret:
        break
    fh, fw = frame.shape[:2]
    detector.setInputSize((fw, fh))

    a = time.perf_counter()
    _, caras = detector.detect(frame)
    t_det += time.perf_counter() - a
    n += 1

    if caras is not None:
        con_rostro += 1
        for c in caras:
            dibujar_deteccion(frame, c[:4], 'Rostro', float(c[-1]), color=(255, 200, 0))
            for i in range(5):   # ojos, nariz y comisuras de la boca
                cv2.circle(frame, (int(c[4 + 2 * i]), int(c[5 + 2 * i])), 3, (0, 0, 255), -1)
        if mejor is None or float(caras[0][-1]) > mejor[0]:
            mejor = (float(caras[0][-1]), frame.copy())

    cv2.imshow('Detector', frame)
    if cv2.waitKey(1) & 0xFF == ord('q') or time.time() - t0 > 20:
        break
cap.release()
cerrar_ventanas()

t = time.time() - t0
print(f'{n} frames en {t:.1f} s ({n/t:.1f} fps), con rostro en {con_rostro} ({100*con_rostro/max(n,1):.0f}%)')
print(f'tiempo promedio del detector: {1000*t_det/max(n,1):.1f} ms por frame')
registrar('6_reto_detector', {'frames': n, 'fps': round(n / t, 1), 'frames_con_rostro': con_rostro,
                              'ms_detector': round(1000 * t_det / max(n, 1), 1),
                              'mejor_confianza': round(mejor[0], 3) if mejor else None})
if mejor:
    cv2.imwrite('resultados/6_detector_rostro.jpg', mejor[1])
    ver(mejor[1], f'detección con más confianza ({mejor[0]:.2f})')

## Resumen

In [ ]:
print(json.dumps(datos, indent=1, ensure_ascii=False, default=str)[:3000])
print()
for raiz, _, archivos in os.walk('resultados'):
    for a in sorted(archivos):
        ruta = os.path.join(raiz, a)
        print(f'  {ruta}  ({os.path.getsize(ruta)//1024} KB)')

Listo, todo queda en `resultados/` (capturas y `datos_lab32.json`).